#  Amazon AI — Product RAG Pipeline

## 1. Project Introduction

Amazon AI is a RAG-powered customer-support assistant designed to answer customer questions using product information and support knowledge.

This notebook focuses on building the **Product Retrieval-Augmented Generation (RAG) Pipeline** using Amazon's cleaned product dataset.

The pipeline will retrieve relevant product information based on a user's natural-language query and provide the retrieved context for grounded LLM responses.

###  Objectives

- Load and inspect the cleaned Amazon product dataset.
- Convert structured product records into meaningful text documents.
- Generate semantic embeddings using Sentence Transformers.
- Build a FAISS vector index for efficient similarity search.
- Retrieve relevant products based on customer queries.
- Evaluate retrieval quality using sample queries.
- Prepare the product retrieval component for integration into the Amazon AI chatbot.

### Product RAG Architecture

Customer Query  
↓  
Query Embedding  
↓  
FAISS Similarity Search  
↓  
Relevant Product Records  
↓  
Retrieved Context → LLM  
↓  
Grounded Customer Response

### Initial Tech Stack

| Component | Technology |
|---|---|
| Data Processing | Pandas |
| Embedding Model | Sentence Transformers |
| Vector Search | FAISS |
| LLM Integration | Groq |
| Interface (later) | Streamlit |

### Development Approach

We will first build and test the pipeline in this notebook. Once the components work correctly, we will modularize the reusable logic into `src/rag.py` for integration with the main application.

**Important:** The quality of the RAG pipeline depends on the quality of the product data, document representation, and retrieval results. We will inspect the dataset before making decisions about document construction.

### **Step by step Impletation.**
#### Imports

In [7]:
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
print("Done")

Done


### Loading datset and basic info

In [8]:
import os
from pathlib import Path

# Walk up until we find the project root (folder containing 'data')
def find_project_root(marker="data"):
    p = Path.cwd()
    for parent in [p, *p.parents]:
        if (parent / marker).exists():
            return parent
    return p

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
print("Project root:", PROJECT_ROOT)
print("Files here:", [f.name for f in PROJECT_ROOT.iterdir()])

Project root: c:\Users\DELL\OneDrive\Documents\Amazn-AI
Files here: ['.env', '.git', '.gitignore', '.venv', 'app.py', 'Assets', 'data', 'LICENSE', 'notebooks', 'README.md', 'requirements.txt', 'src', 'test.py', 'vector_store']


In [9]:
data_path=("data/cleaned/amazon_cleaned.csv")
df = pd.read_csv(data_path)
df.head()

,product_id,product_name,category,discounted_price,actual_price,discount_percentage,rating,rating_count,about_product,user_id,user_name,review_id,review_title,review_content,img_link,product_link
0,B07JW9H4J1,Wayona Nylon Braided USB to Lightning Fast Cha...,Computers&Accessories|Accessories&Peripherals|...,399.0,1099.0,64,4.2,293.0,High Compatibility : Compatible With iPhone 12...,"AG3D6O4STAQKAY2UVGEUV46KN35Q,AHMY5CWJMMK5BJRBB...","Manav,Adarsh gupta,Sundeep,S.Sayeed Ahmed,jasp...","R3HXWT0LRP0NMF,R2AJM3LFTLZHFO,R6AQJGUP6P86,R1K...","Satisfied,Charging is really fast,Value for mo...",Looks durable Charging is fine tooNo complains...,https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Wayona-Braided-WN3LG1-Sy...
1,B098NS6PVG,Ambrane Unbreakable 60W / 3A Fast Charging 1.5...,Computers&Accessories|Accessories&Peripherals|...,199.0,349.0,43,4.0,293.0,"Compatible with all Type C enabled devices, be...","AECPFYFQVRUWC3KGNLJIOREFP5LQ,AGYYVPDD7YG7FYNBX...","ArdKn,Nirbhay kumar,Sagar Viswanathan,Asp,Plac...","RGIQEG07R9HS2,R1SMWZQ86XIN8U,R2J3Y1WL29GWDE,RY...","A Good Braided Cable for Your Type C Device,Go...",I ordered this cable to connect my phone to An...,https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Ambrane-Unbreakable-Char...
2,B096MSW6CT,Sounce Fast Phone Charging Cable & Data Sync U...,Computers&Accessories|Accessories&Peripherals|...,199.0,1899.0,90,3.9,293.0,【 Fast Charger& Data Sync】-With built-in safet...,"AGU3BBQ2V2DDAMOAKGFAWDDQ6QHA,AESFLDV2PT363T2AQ...","Kunal,Himanshu,viswanath,sai niharka,saqib mal...","R3J3EQQ9TZI5ZJ,R3E7WBGK7ID0KV,RWU79XKQ6I1QF,R2...","Good speed for earlier versions,Good Product,W...","Not quite durable and sturdy,https://m.media-a...",https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Sounce-iPhone-Charging-C...
3,B08HDJ86NZ,boAt Deuce USB 300 2 in 1 Type-C & Micro USB S...,Computers&Accessories|Accessories&Peripherals|...,329.0,699.0,53,4.2,293.0,The boAt Deuce USB 300 2 in 1 cable is compati...,"AEWAZDZZJLQUYVOVGBEUKSLXHQ5A,AG5HTSFRRE6NL3M5S...","Omkar dhale,JD,HEMALATHA,Ajwadh a.,amar singh ...","R3EEUZKKK9J36I,R3HJVYCLYOY554,REDECAZ7AMPQC,R1...","Good product,Good one,Nice,Really nice product...","Good product,long wire,Charges good,Nice,I bou...",https://m.media-amazon.com/images/I/41V5FtEWPk...,https://www.amazon.in/Deuce-300-Resistant-Tang...
4,B08CF3B7N1,Portronics Konnect L 1.2M Fast Charging 3A 8 P...,Computers&Accessories|Accessories&Peripherals|...,154.0,399.0,61,4.2,293.0,[CHARGE & SYNC FUNCTION]- This cable comes wit...,"AE3Q6KSUK5P75D5HFYHCRAOLODSA,AFUGIFH5ZAFXRDSZH...","rahuls6099,Swasat Borah,Ajay Wadke,Pranali,RVK...","R1BP4L2HH9TFUP,R16PVJEXKV6QZS,R2UPDB81N66T4P,R...","As good as original,Decent,Good one for second...","Bought this instead of original apple, does th...",https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Portronics-Konnect-POR-1...


In [10]:
df.isnull().sum()

product_id             0
product_name           0
category               0
discounted_price       0
actual_price           0
discount_percentage    0
rating                 0
rating_count           0
about_product          0
user_id                0
user_name              0
review_id              0
review_title           0
review_content         0
img_link               0
product_link           0
dtype: int64

In [11]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1465 entries, 0 to 1464
Data columns (total 16 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   product_id           1465 non-null   str    
 1   product_name         1465 non-null   str    
 2   category             1465 non-null   str    
 3   discounted_price     1465 non-null   float64
 4   actual_price         1465 non-null   float64
 5   discount_percentage  1465 non-null   int64  
 6   rating               1465 non-null   float64
 7   rating_count         1465 non-null   float64
 8   about_product        1465 non-null   str    
 9   user_id              1465 non-null   str    
 10  user_name            1465 non-null   str    
 11  review_id            1465 non-null   str    
 12  review_title         1465 non-null   str    
 13  review_content       1465 non-null   str    
 14  img_link             1465 non-null   str    
 15  product_link         1465 non-null   str    
dtyp

In [12]:
df.columns.tolist()

['product_id',
 'product_name',
 'category',
 'discounted_price',
 'actual_price',
 'discount_percentage',
 'rating',
 'rating_count',
 'about_product',
 'user_id',
 'user_name',
 'review_id',
 'review_title',
 'review_content',
 'img_link',
 'product_link']

In [13]:
df.iloc[0]

product_id                                                    B07JW9H4J1
product_name           Wayona Nylon Braided USB to Lightning Fast Cha...
category               Computers&Accessories|Accessories&Peripherals|...
discounted_price                                                   399.0
actual_price                                                      1099.0
discount_percentage                                                   64
rating                                                               4.2
rating_count                                                       293.0
about_product          High Compatibility : Compatible With iPhone 12...
user_id                AG3D6O4STAQKAY2UVGEUV46KN35Q,AHMY5CWJMMK5BJRBB...
user_name              Manav,Adarsh gupta,Sundeep,S.Sayeed Ahmed,jasp...
review_id              R3HXWT0LRP0NMF,R2AJM3LFTLZHFO,R6AQJGUP6P86,R1K...
review_title           Satisfied,Charging is really fast,Value for mo...
review_content         Looks durable Charging is fi

In [14]:
df.iloc[0].to_dict()

{'product_id': 'B07JW9H4J1',
 'product_name': 'Wayona Nylon Braided USB to Lightning Fast Charging and Data Sync Cable Compatible for iPhone 13, 12,11, X, 8, 7, 6, 5, iPad Air, Pro, Mini (3 FT Pack of 1, Grey)',
 'category': 'Computers&Accessories|Accessories&Peripherals|Cables&Accessories|Cables|USBCables',
 'discounted_price': 399.0,
 'actual_price': 1099.0,
 'discount_percentage': 64,
 'rating': 4.2,
 'rating_count': 293.0,
 'about_product': "High Compatibility : Compatible With iPhone 12, 11, X/XsMax/Xr ,iPhone 8/8 Plus,iPhone 7/7 Plus,iPhone 6s/6s Plus,iPhone 6/6 Plus,iPhone 5/5s/5c/se,iPad Pro,iPad Air 1/2,iPad mini 1/2/3,iPod nano7,iPod touch and more apple devices.|Fast Charge&Data Sync : It can charge and sync simultaneously at a rapid speed, Compatible with any charging adaptor, multi-port charging station or power bank.|Durability : Durable nylon braided design with premium aluminum housing and toughened nylon fiber wound tightly around the cord lending it superior durabilit

##  Product Document & Metadata Creation

### Why Separate Documents and Metadata?

In a RAG pipeline, the document contains the meaningful text used to generate embeddings and perform semantic retrieval.

Metadata stores structured attributes associated with each document, such as product IDs, URLs, and image links.

For Amazon AI, we will separate these two components to improve retrieval quality and preserve exact product information.

### Document
The document contains semantic information such as:
- Product name and category
- Product prices and discount
- Product description
- Ratings and relevant customer reviews

### Metadata
The metadata contains structured product information such as:
- Product ID
- Product URL
- Image URL
- Rating and other useful attributes

Product URLs and image URLs will not be embedded. They will be retrieved from metadata using the matching FAISS result.

This allows Amazon AI to return the original product link instead of asking the LLM to recreate it.

### Design Principle

**Document = Semantic retrieval**

**Metadata = Product identification and exact attribute lookup**

In [15]:
def clean_value(value):
    """Convert missing values into clean strings."""
    if pd.isna(value):
        return ""
    return str(value).strip()


def create_product_documents(row):
     # Semantic document for embedding
    document = f"""
Product Name: {clean_value(row["product_name"])}
Category: {clean_value(row["category"])}
Discounted Price: {clean_value(row["discounted_price"])}
Actual Price: {clean_value(row["actual_price"])}
Discount: {clean_value(row["discount_percentage"])}
Rating: {clean_value(row["rating"])}
Rating Count: {clean_value(row["rating_count"])}
Product Description:{clean_value(row["about_product"])}
Customer Review Title:{clean_value(row["review_title"])}
Customer Review:{clean_value(row["review_content"])}
""".strip()

    # Metadata stored separately
    metadata = {
        "product_id": clean_value(row["product_id"]),
        "product_name": clean_value(row["product_name"]),
        "product_link": clean_value(row["product_link"]),
        "img_link": clean_value(row["img_link"]),
        "rating": clean_value(row["rating"]),
        "rating_count": clean_value(row["rating_count"]),
        "discounted_price": clean_value(row["discounted_price"]),
        "actual_price": clean_value(row["actual_price"]),
        "category": clean_value(row["category"])
    }

    return {
        "document": document,
        "metadata": metadata
    }
print("Done")

Done


#### Test 

In [16]:
sample_product = create_product_documents(df.iloc[0])

print("Documents:\n")
print(sample_product["document"])

print("\n metadata:\n")
sample_product["metadata"]

Documents:

Product Name: Wayona Nylon Braided USB to Lightning Fast Charging and Data Sync Cable Compatible for iPhone 13, 12,11, X, 8, 7, 6, 5, iPad Air, Pro, Mini (3 FT Pack of 1, Grey)
Category: Computers&Accessories|Accessories&Peripherals|Cables&Accessories|Cables|USBCables
Discounted Price: 399.0
Actual Price: 1099.0
Discount: 64
Rating: 4.2
Rating Count: 293.0
Product Description:High Compatibility : Compatible With iPhone 12, 11, X/XsMax/Xr ,iPhone 8/8 Plus,iPhone 7/7 Plus,iPhone 6s/6s Plus,iPhone 6/6 Plus,iPhone 5/5s/5c/se,iPad Pro,iPad Air 1/2,iPad mini 1/2/3,iPod nano7,iPod touch and more apple devices.|Fast Charge&Data Sync : It can charge and sync simultaneously at a rapid speed, Compatible with any charging adaptor, multi-port charging station or power bank.|Durability : Durable nylon braided design with premium aluminum housing and toughened nylon fiber wound tightly around the cord lending it superior durability and adding a bit to its flexibility.|High Security Level 

{'product_id': 'B07JW9H4J1',
 'product_name': 'Wayona Nylon Braided USB to Lightning Fast Charging and Data Sync Cable Compatible for iPhone 13, 12,11, X, 8, 7, 6, 5, iPad Air, Pro, Mini (3 FT Pack of 1, Grey)',
 'product_link': 'https://www.amazon.in/Wayona-Braided-WN3LG1-Syncing-Charging/dp/B07JW9H4J1/ref=sr_1_1?qid=1672909124&s=electronics&sr=1-1',
 'img_link': 'https://m.media-amazon.com/images/W/WEBP_402378-T1/images/I/51UsScvHQNL._SX300_SY300_QL70_FMwebp_.jpg',
 'rating': '4.2',
 'rating_count': '293.0',
 'discounted_price': '399.0',
 'actual_price': '1099.0',
 'category': 'Computers&Accessories|Accessories&Peripherals|Cables&Accessories|Cables|USBCables'}

In [17]:
product_record=df.apply(create_product_documents,axis=1).tolist()
print(f"Created {len(product_record)} documents Sucessfully.")
product_record[1]

Created 1465 documents Sucessfully.


{'document': "Product Name: Ambrane Unbreakable 60W / 3A Fast Charging 1.5m Braided Type C Cable for Smartphones, Tablets, Laptops & other Type C devices, PD Technology, 480Mbps Data Sync, Quick Charge 3.0 (RCT15A, Black)\nCategory: Computers&Accessories|Accessories&Peripherals|Cables&Accessories|Cables|USBCables\nDiscounted Price: 199.0\nActual Price: 349.0\nDiscount: 43\nRating: 4.0\nRating Count: 293.0\nProduct Description:Compatible with all Type C enabled devices, be it an android smartphone (Mi, Samsung, Oppo, Vivo, Realme, OnePlus, etc), tablet, laptop (Macbook, Chromebook, etc)|Supports Quick Charging (2.0/3.0)|Unbreakable – Made of special braided outer with rugged interior bindings, it is ultra-durable cable that won’t be affected by daily rough usage|Ideal Length – It has ideal length of 1.5 meters which is neither too short like your typical 1meter cable or too long like a 2meters cable|Supports maximum 3A fast charging and 480 Mbps data transfer speed|6 months manufacturer

#### Loading embedding model.

In [18]:
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
print("Embedding model loaded successfully")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully


#### Seperate metedata and documents

In [19]:
documents = [
    record["document"]
    for record in product_record
]

metadata = [
    record["metadata"]
    for record in product_record
]

print(f"We have {len(documents)} documents")
print(f"We have {len(metadata)} metadata records")

We have 1465 documents
We have 1465 metadata records


In [20]:
print("Document sample:\n", documents[0])
print("\nMetadata sample:\n", metadata[0])

Document sample:
 Product Name: Wayona Nylon Braided USB to Lightning Fast Charging and Data Sync Cable Compatible for iPhone 13, 12,11, X, 8, 7, 6, 5, iPad Air, Pro, Mini (3 FT Pack of 1, Grey)
Category: Computers&Accessories|Accessories&Peripherals|Cables&Accessories|Cables|USBCables
Discounted Price: 399.0
Actual Price: 1099.0
Discount: 64
Rating: 4.2
Rating Count: 293.0
Product Description:High Compatibility : Compatible With iPhone 12, 11, X/XsMax/Xr ,iPhone 8/8 Plus,iPhone 7/7 Plus,iPhone 6s/6s Plus,iPhone 6/6 Plus,iPhone 5/5s/5c/se,iPad Pro,iPad Air 1/2,iPad mini 1/2/3,iPod nano7,iPod touch and more apple devices.|Fast Charge&Data Sync : It can charge and sync simultaneously at a rapid speed, Compatible with any charging adaptor, multi-port charging station or power bank.|Durability : Durable nylon braided design with premium aluminum housing and toughened nylon fiber wound tightly around the cord lending it superior durability and adding a bit to its flexibility.|High Security 

#### Generating embeddings to documents

In [21]:
embeddings=embedding_model.encode(
    documents,
    convert_to_numpy=True,
    show_progress_bar=True
)
embeddings=embeddings.astype("float32")
print(f"Embedding shape:{embeddings.shape}")

Batches:   0%|          | 0/46 [00:00<?, ?it/s]

Embedding shape:(1465, 384)


#### Store embeddings to faiss

In [22]:
faiss.normalize_L2(embeddings)
embedding_dimensions=embeddings.shape[1]
vector_index=faiss.IndexFlatIP(embedding_dimensions)
vector_index.add(embeddings)
print("Total Vectors stored",vector_index.ntotal)


Total Vectors stored 1465


#### Save the index and metadata together

In [ ]:
import json
from pathlib import Path


vector_store_path = Path("../vector_store")
vector_store_path.mkdir(parents=True, exist_ok=True)


faiss.write_index(
    vector_index,
    str(vector_store_path / "product_index.faiss")
)

with open(vector_store_path / "product_metadata.json", "w", encoding="utf-8") as file:
    json.dump(metadata, file, ensure_ascii=False, indent=4)


with open(vector_store_path / "product_documents.json", "w", encoding="utf-8") as file:
    json.dump(documents, file, ensure_ascii=False, indent=4)

print("FAISS index, metadata, and documents saved!")

FAISS index, metadata, and documents saved!


#### Making retrive function

In [24]:
def retrieve_products(query, top_k=5):
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")
    faiss.normalize_L2(query_embedding)
    distances, indices = vector_index.search(
        query_embedding,
        top_k
    )

    results = []
    for score, idx in zip(distances[0], indices[0]):
        if idx == -1:
            continue
        result = {
            "document": documents[idx],
            "metadata": metadata[idx],
            "similarity_score": float(score)
        }
        results.append(result)
    return results

### Test the query

In [25]:
results = retrieve_products(
    "Samsung Type C charging cable",
    top_k=5
)

for i, result in enumerate(results, start=1):

    print(f"\n--- Result {i} ---")

    print("Product:", result["metadata"]["product_name"])
    print("Product ID:", result["metadata"]["product_id"])
    print("Similarity:", result["similarity_score"])

    print("\nDocument:")
    print(result["document"])

    print("\nProduct Link:")
    print(result["metadata"]["product_link"])


--- Result 1 ---
Product: Samsung Original Type C to C Cable - 3.28 Feet (1 Meter), White
Product ID: B008FWZGSG
Similarity: 0.7370076179504395

Document:
Product Name: Samsung Original Type C to C Cable - 3.28 Feet (1 Meter), White
Category: Computers&Accessories|Accessories&Peripherals|Cables&Accessories|Cables|USBCables
Discounted Price: 599.0
Actual Price: 599.0
Discount: 0
Rating: 4.3
Rating Count: 355.0
Product Description:USB Type-C to Type-C cable with universal compatibility|1m Length & Reversible design|High Speed Data/Charging with USB 2.0
Customer Review Title:Good,Genuine product,Not first charge this cabil,,Nice cable but known durability,Got a used type c to c cable and the xable was damaged,The product is Genuine and Good,Abdul kadir,Good
Customer Review:Buy it,Received in good condition and is working properly,This price cabil not good,  and also short,I am using from 2 months and charging the phone in good way and it's really value for money.. i recommend this cable.

#### Test layer

In [26]:
import importlib
import src.llm
importlib.reload(src.llm)
from src.llm import ask_amazon as generate_answer
print("LLM module reloaded!")

Done All works perfectly
Done All works perfectly
LLM module reloaded!


#### Final LLM layer

In [27]:

def amazon_rag(query):
    results = retrieve_products(query)
    context = "\n\n -- \n\n".join(
        f"""
{result["document"]}

Product ID: {result["metadata"]["product_id"]}
Product Link: {result["metadata"]["product_link"]}
Product Image: {result["metadata"]["img_link"]}
"""
        for result in results
    )
    answer = generate_answer(query, context)
    return answer

In [28]:
response = amazon_rag("Suggest a Samsung Type C cable")
print(response)

Here’s a Samsung‑branded Type‑C cable that matches what you’re looking for:

**Samsung Original Type‑C to Type‑C Cable – 1 m (3.28 ft), White**  
- **Length:** 1 meter (3.28 ft) – short enough for everyday use but long enough to reach most ports.  
- **Compatibility:** Universal USB‑C, works with any device that has a USB‑C port (phones, tablets, laptops, etc.).  
- **Charging & Data:** Supports high‑speed data transfer and 3 A charging (USB 2.0 spec).  
- **Build:** Reversible design, durable cable, and genuine Samsung quality.  
- **Rating:** 4.3 ★ (355 reviews) – many users report reliable charging and good durability.  

**Product link:**  
[Samsung Original Type‑C to Type‑C Cable – 1 m, White](https://www.amazon.in/Samsung-Original-Type-Cable-Meter/dp/B008FWZGSG/ref=sr_1_271?qid=1672903008&s=computers&sr=1-271)

This is the only Samsung‑branded Type‑C cable listed in the provided context, so it’s the best match for your request.


In [29]:
from pathlib import Path
PROJECT_ROOT = Path.cwd()
print("Current directory:", PROJECT_ROOT)
# Check expected vector store directory
vector_store = PROJECT_ROOT / "vector_store"

print("Vector store exists:", vector_store.exists())

if vector_store.exists():
    print("Files inside vector_store:")
    for file in vector_store.iterdir():
        print(file.name)

# Search project for any FAISS index
print("\nSearching for FAISS files...")

for file in PROJECT_ROOT.rglob("*.faiss"):
    print(file)

Current directory: c:\Users\DELL\OneDrive\Documents\Amazn-AI
Vector store exists: True
Files inside vector_store:
faiss_index

Searching for FAISS files...


In [30]:

from pathlib import Path

# Notebook is inside: project_root/notebooks/product-rag
PROJECT_ROOT = Path.cwd().parents[1]

VECTOR_STORE_DIR = PROJECT_ROOT / "vector_store"

print("Project root:", PROJECT_ROOT)
print("Vector store path:", VECTOR_STORE_DIR)
print("Vector store exists:", VECTOR_STORE_DIR.exists())

if VECTOR_STORE_DIR.exists():
    print("\nFiles inside vector_store:")
    for file in VECTOR_STORE_DIR.iterdir():
        print(file.name)

print("\nSearching entire project for FAISS index files:")

faiss_files = list(PROJECT_ROOT.rglob("*.faiss"))

if faiss_files:
    for file in faiss_files:
        print(file)
else:
    print("No .faiss files found.")

Project root: c:\Users\DELL\OneDrive
Vector store path: c:\Users\DELL\OneDrive\vector_store
Vector store exists: False

Searching entire project for FAISS index files:
c:\Users\DELL\OneDrive\Documents\vector_store\product_index.faiss
